In [1]:
from libero.libero import benchmark
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from openpi_client import image_tools

import pathlib
import numpy as np
import math
import os

CAMERA_NAMES = ["agentview", "birdview", "robot0_eye_in_hand", "sideview", "canonical_frontview"]

def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_names": CAMERA_NAMES}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

def _get_empty_env(task, env):
    import robosuite
    dataset_file = os.path.join(get_libero_path("datasets"), f"{task.problem_folder}/{task.name}_demo.hdf5")
    import h5py
    import json
    f = h5py.File(dataset_file, "r")
    env_meta = json.loads(f["data"].attrs["env_args"])
    f.close()
    empty_env_kwargs = env_meta['env_kwargs'].copy()
    empty_env_kwargs['env_name'] = "SingleArmEmptyEnv"
    empty_env_kwargs['hard_reset'] = False
    empty_env_kwargs['ignore_done'] = True
    empty_env_kwargs['has_offscreen_renderer'] = False
    empty_env_kwargs['has_renderer'] = False
    empty_env_kwargs['use_camera_obs'] = False
    empty_env_kwargs['camera_names'] = CAMERA_NAMES
    empty_env_kwargs['camera_heights'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['camera_widths'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['robots'] = [type(robot.robot_model).__name__ for robot in env.robots]
    empty_env = robosuite.make(**empty_env_kwargs)
    empty_env.copy_env_model(env)
    return empty_env


task_suite_name = "libero_10"
task_id = 4
seed = 1
from vlm_agent import VLMAgent
from vlm_utils import *
agent = VLMAgent(task_suite_name, task_id)

LIBERO_ENV_RESOLUTION = 224
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[task_suite_name]()
task = task_suite.get_task(task_id)
initial_states = task_suite.get_task_init_states(task_id)
env, task_description = _get_libero_env(task, LIBERO_ENV_RESOLUTION, seed)
empty_env = _get_empty_env(task, env)
print(f"Task description: {task_description}")

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/scripts/setup_macros.py (macros.py:55)


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Task description: put the white mug on the left plate and put the yellow and white mug on the right plate


In [2]:
from wm_client.client import WMClient
from wm_client.wm_env import WMEnv
host = "0.0.0.0"
port = 7880
wm_client = WMClient(host, port)
wm_env = WMEnv(env, empty_env, wm_client)

Connecting to ws://0.0.0.0:7880...
Connected to ws://0.0.0.0:7880


In [3]:
from dp_utils import embed_lang
subtasks = libero10_subtask_map[task_id]
avail_task_suite = benchmark_dict["libero_90"]()
subtask_embeddings = []
subtask_descriptions = []
for subtask_id in subtasks:
    subtask = avail_task_suite.get_task(subtask_id)
    subtask_description = subtask.language
    print(f"Subtask description: {subtask_description}")
    subtask_embedding = embed_lang(subtask_description)
    subtask_embeddings.append(subtask_embedding)
    subtask_descriptions.append(subtask_description)

/n/holylabs/ydu_lab/Lab/zhangxiangcheng/miniconda3/envs/libero_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robomimic/robomimic/scripts/setup_macros.py
)
Subtask description: put the white mug on the left plate
Loaded language embed from cache.
Subtask description: put the yellow and white mug on the right plate
Loaded language embed from cache.


In [4]:
from dp_utils import load_checkpoint
import robosuite.utils.transform_utils as T
checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.02.27/09.11.46_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0014-test_mean_score=1.000.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.02.27/20.14.00_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0022-test_mean_score=1.000.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.05/06.14.04_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0022-test_mean_score=1.000.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.08/09.41.02_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0300-test_mean_score=1.000.ckpt"
policy, cfg = load_checkpoint(checkpoint_path)
policy = policy.to("cuda")
import torch
def to_torch(image):
    image = image_tools.resize_with_pad(image, 128, 128)
    return np.moveaxis(image[::-1], -1, 0) / 255.0
def policy_fn(obs, subtask_id=0):
    np_obs_dict = dict(obs)
    if "lang_embed" in cfg.shape_meta.obs:
        np_obs_dict["lang_embed"] = subtask_embeddings[subtask_id]
    obs_keys = cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    np_obs_dict = {k: to_torch(v) if "image" in k else v for k, v in np_obs_dict.items()}
    obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0).unsqueeze(0) for k, v in np_obs_dict.items()}
    with torch.no_grad():
        action_dict = policy.predict_action(obs_dict)
    np_action_dict = {k: v.cpu().numpy() for k, v in action_dict.items()}
    action = np_action_dict['action_pred'][0]
    return action

# idm_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.02/07.30.54_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0100-val_loss=0.034.ckpt"
# idm_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.01/20.20.00_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0200-val_loss=0.058.ckpt"
idm_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.12/01.36.19_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0020-val_loss=0.025.ckpt"
idm, idm_cfg = load_checkpoint(idm_checkpoint_path)
idm = idm.to("cuda")
def idm_fn(obs, target_pos, target_quat=None):
    np_obs_dict = dict(obs)
    obs_keys = idm_cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    delta_obs_dict = {"robot0_eef_pos": target_pos - obs['robot0_eef_pos']}
    if target_quat is not None:
        delta_obs_dict['robot0_eef_quat'] = T.quat_distance(target_quat, obs['robot0_eef_quat'])
    obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in np_obs_dict.items()}
    delta_obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in delta_obs_dict.items()}
    with torch.no_grad():
        action_dict = idm.predict_action(obs_dict, delta_obs_dict)
    np_pred_action = action_dict['action_pred'].cpu().numpy()[0]
    return np_pred_action

# idm_2_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.02/07.30.54_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0100-val_loss=0.034.ckpt"
idm_2_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.11/21.52.52_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0040-val_loss=0.026.ckpt"
idm_2, idm_2_cfg = load_checkpoint(idm_2_checkpoint_path)
idm_2 = idm_2.to("cuda")
def idm_fn_2(obs, target_pos, target_quat=None):
    np_obs_dict = dict(obs)
    obs_keys = idm_2_cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    delta_obs_dict = {"robot0_eef_pos": target_pos - obs['robot0_eef_pos']}
    if target_quat is not None:
        delta_obs_dict['robot0_eef_quat'] = T.quat_distance(target_quat, obs['robot0_eef_quat'])
    obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in np_obs_dict.items()}
    delta_obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in delta_obs_dict.items()}
    with torch.no_grad():
        action_dict = idm_2.predict_action(obs_dict, delta_obs_dict)
    np_pred_action = action_dict['action_pred'].cpu().numpy()[0]
    return np_pred_action


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_pos', 'lang_embed', 'robot0_eef_quat', 'robot0_gripper_qpos']
using obs modality: rgb with keys: ['agentview_image', 'robot0_eye_in_hand_image']
using obs modality: depth with keys: []
using obs modality: scan with keys: []


In [7]:
wm_env.reset()
num_steps = 0
subtask_id = 0
replay_images = []
agent_monitor = False
obs = env.set_init_state(initial_states[seed])
for t in range(60):
    obs, reward, done, info = wm_env.step(LIBERO_DUMMY_ACTION)
agent.start_episode(obs)

In [8]:
target_point = obs['robot0_eef_pos']
target_quat = obs['robot0_eef_quat']
print(f"Initial target point: {target_point}, target quat: {target_quat}")

Initial target point: [-0.06016326 -0.00941683  0.68037633], target quat: [ 9.99592615e-01  6.36693676e-05 -2.85408300e-02  1.42540620e-04]


In [11]:
action_chunk = idm_fn(obs, target_point)
action_chunk[:, -1] = -1
for _ in range(10):
    obs, reward, done, info = wm_env.step(LIBERO_DUMMY_ACTION)
    replay_images.append(obs["agentview_image"][::-1])
for action in action_chunk:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])

/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [12]:
import tqdm
pbar = tqdm.tqdm(total=500, desc="Executing policy")
while not done and num_steps < 500:
    agent.cache_obs(obs)
    action_chunk = policy_fn(obs, subtask_id=subtask_id)[:10]
    # num_noop = sum([is_noop(action, obs, threshold=0.08) for action in action_chunk])
    # if num_noop >= 5 and subtask_id == 1:
    #     print(f"Detected {num_noop} consecutive no-op actions at t={num_steps}, stopping execution to prevent potential infinite loop.")
    #     break
    for action in (action_chunk):
        obs, reward, done, info = wm_env.step(action)
        replay_images.append(obs["agentview_image"][::-1])
    pbar.update(10)
    num_steps += 10

    if subtask_id == 0 and num_steps >= 70 and agent.verify_subtask_completion(subtask_descriptions, obs):
        subtask_id = 1
        # action_chunk = policy_fn(obs, subtask_id=subtask_id)[:10]
        # for action in (action_chunk):
        #     obs, reward, done, info = wm_env.step(action)
        #     replay_images.append(obs["agentview_image"][::-1])
        # pbar.update(10)
        # num_steps += 10
        break
        
    

Executing policy:  14%|███▋                      | 70/500 [00:07<00:45,  9.44it/s]

In [7]:
agent.start_mpc(obs)
agent_actions = agent.get_action_proposal()

API call took 72.98 seconds.
Thinking tokens used: 2282
output tokens used: 313
API response: Based on the current state and the task instructions, the robot has already grasped the white mug and positioned it over the left plate. The next steps involve releasing the white mug on the left plate, then moving to and grasping the yellow and white mug to place it on the right plate.

Here is the proposed action sequence:

```json
[
    {
        "action": "RELEASE",
        "parameters": {}
    },
    {
        "action": "MOVE",
        "parameters": {
            "frontview": {"x": 648, "y": 571},
            "topview": {"x": 571, "y": 494},
            "sideview": {"x": 598, "y": 608}
        }
    },
    {
        "action": "GRASP",
        "parameters": {}
    },
    {
        "action": "MOVE",
        "parameters": {
            "frontview": {"x": 915, "y": 670},
            "topview": {"x": 719, "y": 534},
            "sideview": {"x": 724, "y": 737}
        }
    },
    {
        "a

In [ ]:
gripper_action = -1
for action_dict in agent_actions:
    if action_dict["action"] == "RELEASE":
        for _ in range(10):
            obs, reward, done, info = wm_env.step(LIBERO_DUMMY_ACTION)
            replay_images.append(obs["agentview_image"][::-1])
        pbar.update(10)
        num_steps += 10
        gripper_action = -1
    if action_dict["action"] == "MOVE":
        plot_coordinates_on_image(obs, action_dict['parameters'])
        target_point = generate_3d_point(action_dict['parameters'], empty_env.get_camera_info())
        target_quat = None
        action_chunk = idm_fn(obs, target_point)
        action_chunk = update_gripper_action(action_chunk, gripper_action)
        break

NameError: name 'agent_actions' is not defined

In [ ]:
import imageio
candidate_quats = generate_rotation_candidates()
candidate_actions = []
candidate_obs = []
for i, quat in enumerate(candidate_quats):
    action_chunk = update_gripper_action(idm_fn(obs, target_point, target_quat=quat), gripper_action)
    candidate_actions.append(action_chunk)
    with wm_env.simulation():
        pred_obs = wm_env.simulate(action_chunk)
        candidate_obs.append(pred_obs['future_obs'])
    imageio.mimwrite(f"test_dp_output_candidate{i}.mp4", pred_obs['WMPredictionOutput'].full_video, fps=20)
best_id = agent.verify_rotation(candidate_obs)
action_chunk = candidate_actions[best_id]
wm_agent_obs = candidate_obs[best_id]
target_quat = candidate_quats[best_id]


TypeError: generate_rotation_candidates() takes 0 positional arguments but 1 was given

In [14]:
import imageio
with wm_env.simulation():
    pred_obs = wm_env.simulate(action_chunk)
    wm_agent_obs = pred_obs['future_obs']
imageio.mimwrite('test_dp_output_wm_1.mp4', pred_obs['WMPredictionOutput'].full_video, fps=20)

In [10]:
traj_response = agent.optimize_trajectory(wm_agent_obs)

API call took 28.57 seconds.
Thinking tokens used: 459
output tokens used: 309
API response for trajectory optimization: To complete the task of placing the yellow and white mug on the right plate, the robot needs to move its gripper from the left side of the table (where it just placed the white mug) to the location of the yellow and white mug. 

Analyzing the provided trajectory images:
1. The robot is moving the gripper horizontally across the table towards the right.
2. In the path between the left plate and the yellow/white mug, there is a tall red mug standing in the center.
3. Observing the middle frames of the trajectory (especially frames 10 through 18), the bottom of the robot's gripper is moving at a height very close to the top rim of the red mug. This creates a high risk of a collision that could knock over the red mug.
4. To ensure a safe and collision-free path, the gripper trajectory should be adjusted upwards to provide sufficient clearance over the red mug.

According

In [11]:
midpoint_adjustment = optimize_trajectory(traj_response)
# midpoint_adjustment = 0
midpoint_obs = wm_agent_obs[20]
midpoint_obs['robot0_eef_pos'] += midpoint_adjustment
action_chunk = update_gripper_action(idm_fn_2(obs, midpoint_obs['robot0_eef_pos'], midpoint_obs['robot0_eef_quat']), gripper_action)
for action in action_chunk:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])
    pbar.update(1)
    num_steps += 1
agent.cache_obs(obs)
action_chunk = idm_fn_2(obs, target_point, target_quat=target_quat)
action_chunk = update_gripper_action(action_chunk, gripper_action)


Executing policy:  18%|████▎                   | 91/500 [05:50<1:25:50, 12.59s/it]/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]
Executing policy:  22%|█████▌                   | 110/500 [05:50<25:44,  3.96s/it]

In [13]:
height_response = agent.optimize_height(wm_agent_obs)
endpoint_adjustment = np.array([0, 0, height_response]) * 0.03
target_point += endpoint_adjustment
action_chunk = idm_fn_2(obs, target_point, target_quat=target_quat)
action_chunk = update_gripper_action(action_chunk, gripper_action)

API call took 26.11 seconds.
Thinking tokens used: 1397
output tokens used: 15
API response for gripper height optimization: ```json
{
    "z": 1
}
```


In [20]:
endpoint_response = agent.optimize_endpoint(wm_agent_obs)

API call took 102.48 seconds.
Thinking tokens used: 6649
output tokens used: 504
API response for endpoint optimization: To achieve the task of putting the yellow and white mug on the right plate, the robot first needs to grasp it. Based on the provided images and the task history, here is the analysis:

1.  **Understand the Goal**: The robot has already placed the white mug on the left plate. Now, it is positioning itself to grasp the yellow and white mug (located on the right side of the table from the front view) to eventually place it on the right plate.

2.  **Frontview Analysis (Y and Z axes)**:
    *   The gripper is currently positioned above the yellow and white mug.
    *   In the horizontal direction (Y-axis), the gripper is shifted to the left of the mug's center. However, the right jaw of the gripper is already inside the mug's rim, while the left jaw is outside. According to **Guideline 3**, we should not adjust the Y direction if one jaw is already inside the cup. Theref

In [21]:
endpoint_adjustment = np.array([endpoint_response['x'] * 0.02, endpoint_response['y'] * 0.02, endpoint_response['z'] * 0.02])
target_point += endpoint_adjustment

In [22]:
candidate_points = generate_candidates(target_point)

In [23]:
candidate_obs = []
candidate_actions = []
for i, candidate_point in enumerate(candidate_points):
    with wm_env.simulation():
        candidate_action = update_gripper_action(idm_fn_2(obs, candidate_point, target_quat), gripper_action)
        candidate_actions.append(candidate_action)
        pred_obs = wm_env.simulate(candidate_action)
        next_action_chunk = policy_fn(pred_obs['future_obs'][-1], subtask_id=1)[:20]
        next_obs = wm_env.simulate(next_action_chunk)
        next_action_chunk = policy_fn(next_obs['future_obs'][-1], subtask_id=1)[:20]
        next_obs = wm_env.simulate(next_action_chunk)
    imageio.mimwrite(f'test_dp_output_candidate{i}.mp4', next_obs['WMPredictionOutput'].full_video, fps=20)
    candidate_obs.append(next_obs['future_obs'])

/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [24]:
# frontview_ranking = agent.rank_images_frontview(candidate_obs)
wristview_ranking = agent.rank_images_wristview(candidate_obs)

API call took 92.17 seconds.
Thinking tokens used: 4161
output tokens used: 20
API response for wristview image ranking: ```json
[3, 0, 4, 1, 2]
```


In [26]:
new_point = (candidate_points[wristview_ranking[0]] + candidate_points[wristview_ranking[1]]) / 2
noise = [[0, 0, 0.03], [0, 0, -0.03,], [0, 0.03,0], [0, -0.03, 0]]
candidate_points = [candidate_points[wristview_ranking[0]],] + [new_point + np.array(n) for n in noise]

In [25]:
action_chunk = candidate_actions[3]

In [26]:
for action in action_chunk:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])

In [ ]:
import imageio
imageio.mimwrite('test_dp_output.mp4', replay_images, fps=20)
replay_images = []

/n/holylabs/ydu_lab/Lab/zhangxiangcheng/miniconda3/envs/libero_env/lib/python3.10/site-packages/imageio_ffmpeg/_utils.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_filename
/n/holylabs/ydu_lab/Lab/zhangxiangcheng/miniconda3/envs/libero_env/lib/python3.10/site-packages/pkg_resources/__init__.py:3147: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)


Executing policy:  14%|███▎                    | 70/500 [28:54<2:57:33, 24.78s/it]